<a href="https://colab.research.google.com/github/rounak393/clab/blob/main/resnet_hmu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ["KAGGLE_API_TOKEN"]='KGAT_c03d989b55c966d18c971a92b023645b'

In [2]:
!kaggle datasets download -d britikak/busi-dataset

Dataset URL: https://www.kaggle.com/datasets/britikak/busi-dataset
License(s): unknown
100% 195M/195M [00:02<00:00, 86.8MB/s]



In [3]:
!unzip -q busi-dataset.zip -d busi_dataset

In [10]:
import os
import copy
import torch
import random
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from PIL import Image
import albumentations as A
from tqdm import tqdm

torch.backends.cudnn.benchmark = True

BASE_DIR = "/content/busi_dataset/Dataset_BUSI_with_GT"
CLASSES = ["benign", "malignant"]
IMG_SIZE = 256

BATCH_SIZE = 4
EPOCHS = 100


torch.manual_seed(999)
random.seed(999)
np.random.seed(999)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.GaussianBlur(blur_limit=(3, 7), sigma_limit=0.5, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [6]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)

class DCSAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.smooth = nn.AvgPool2d(3, stride=1, padding=1)
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, max(channels // 8, 4), 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(channels // 8, 4), channels, 1, bias=False),
            nn.Sigmoid()
        )
        self.sa = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.Sigmoid())

    def forward(self, x):
        details = x - self.smooth(x)
        x = x + details
        x = x * self.ca(x)
        sa_in = torch.cat([torch.mean(x, dim=1, keepdim=True), torch.max(x, dim=1, keepdim=True)[0]], dim=1)
        return x * self.sa(sa_in)

class HMAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        inter = channels // 4
        self.reduce = nn.Conv2d(channels, inter * 3, 1, bias=False)
        self.d1 = nn.Conv2d(inter, inter, 3, padding=1, dilation=1, bias=False)
        self.d3 = nn.Conv2d(inter, inter, 3, padding=3, dilation=3, bias=False)
        self.d5 = nn.Conv2d(inter, inter, 3, padding=5, dilation=5, bias=False)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.pool_conv = nn.Conv2d(channels, inter, 1, bias=False)
        self.fuse = nn.Conv2d(inter * 4, channels, 1, bias=False)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // 8, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 8, channels, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b1, b2, b3 = torch.split(self.reduce(x), x.shape[1] // 4, dim=1)
        b1, b2, b3 = self.d1(b1), self.d3(b2), self.d5(b3)
        b4 = F.interpolate(self.pool_conv(self.pool(x)), size=x.shape[2:], mode='bilinear', align_corners=False)
        fused = self.fuse(torch.cat([b1, b2, b3, b4], dim=1))
        return fused * self.se(fused)

In [7]:
class ResNet50HMUNet(nn.Module):
    def __init__(self, out_channels=1):
        super().__init__()
        # Load pre-trained ResNet50
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        # 1. ENCODER (Splicing the ResNet Layers)
        self.e1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu) # Out: 64 channels (128x128)
        self.maxpool = resnet.maxpool
        self.e2 = resnet.layer1 # Out: 256 channels (64x64)
        self.e3 = resnet.layer2 # Out: 512 channels (32x32)
        self.e4 = resnet.layer3 # Out: 1024 channels (16x16)

        # 2. BOTTLENECK
        self.bottleneck = resnet.layer4 # Out: 2048 channels (8x8)
        self.hmam = HMAM(2048)

        # 3. EDGE SKIP FILTERS
        self.dcsam1 = DCSAM(64)
        self.dcsam2 = DCSAM(256)
        self.dcsam3 = DCSAM(512)
        self.dcsam4 = DCSAM(1024)

        # 4. DECODER
        self.up4 = nn.ConvTranspose2d(2048, 1024, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(2048, 1024)

        self.up3 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(1024, 512)

        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(512, 256)

        self.up1 = nn.ConvTranspose2d(256, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        # Final upsample step to restore original 256x256 image size
        self.up0 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec0 = DoubleConv(32, 32)
        self.final_conv = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder Pass
        e1 = self.e1(x)
        e2 = self.e2(self.maxpool(e1))
        e3 = self.e3(e2)
        e4 = self.e4(e3)

        # Bottleneck Pass
        b = self.hmam(self.bottleneck(e4))

        # Decoder Pass with DCSAM filtered skips
        d4 = self.dec4(torch.cat([self.dcsam4(e4), self.up4(b)], dim=1))
        d3 = self.dec3(torch.cat([self.dcsam3(e3), self.up3(d4)], dim=1))
        d2 = self.dec2(torch.cat([self.dcsam2(e2), self.up2(d3)], dim=1))
        d1 = self.dec1(torch.cat([self.dcsam1(e1), self.up1(d2)], dim=1))

        # Final sizing
        d0 = self.dec0(self.up0(d1))
        return self.final_conv(d0)

In [8]:
class BUSISegmentationDataset(Dataset):
    def __init__(self, base_dir, classes, transform=None):
        self.samples = []
        self.transform = transform
        for cls in classes:
            cls_dir = os.path.join(base_dir, cls)
            images = [f for f in os.listdir(cls_dir) if f.endswith(".png") and "_mask" not in f]
            for img_name in images:
                img_path = os.path.join(cls_dir, img_name)
                base_name = img_name.replace(".png", "")
                mask_files = [f for f in os.listdir(cls_dir) if f.startswith(base_name + "_mask") and f.endswith(".png")]
                if not mask_files: continue
                self.samples.append((img_path, [os.path.join(cls_dir, f) for f in mask_files]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_paths = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        combined_mask = np.zeros(image.shape[:2], dtype=np.uint8)
        for mpath in mask_paths:
            mask = np.array(Image.open(mpath).convert("L"))
            combined_mask = np.logical_or(combined_mask, (mask > 0).astype(np.uint8))
        combined_mask = combined_mask.astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=combined_mask)
            image, combined_mask = augmented["image"], augmented["mask"]

        return torch.from_numpy(image).permute(2, 0, 1).float(), torch.from_numpy(combined_mask).unsqueeze(0).float()

full_dataset = BUSISegmentationDataset(BASE_DIR, classes=CLASSES, transform=None)
indices = list(range(len(full_dataset)))
np.random.shuffle(indices)

train_size, val_size = int(0.8 * len(full_dataset)), int(0.1 * len(full_dataset))
train_dataset = Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=train_transform), indices[val_size:train_size + val_size])
val_dataset = Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=val_transform), indices[:val_size])
test_dataset = Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=val_transform), indices[train_size + val_size:])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class StrictBCEDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        preds = torch.sigmoid(logits).view(-1)
        targets_f = targets.view(-1)
        inter = (preds * targets_f).sum()
        dice_loss = 1 - (2 * inter + self.smooth) / (preds.sum() + targets_f.sum() + self.smooth)
        return 0.2 * bce_loss + 0.8 * dice_loss

def strict_dice_coef(y_true, logits, smooth=1e-5):
    y_pred = (torch.sigmoid(logits) > 0.5).float().view(-1)
    y_true_f = y_true.view(-1)
    inter = (y_true_f * y_pred).sum()
    return (2. * inter + smooth) / (y_true_f.sum() + y_pred.sum() + smooth)

In [11]:
# Initializing the new Hybrid Model
model = ResNet50HMUNet(out_channels=1).to(device)

criterion = StrictBCEDiceLoss()
# Note: Weight decay added here to fight overfitting
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda')

best_val_dice = 0.0
best_model_weights = None
best_epoch = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    train_dice = 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        with torch.no_grad():
            train_dice += strict_dice_coef(masks, logits).item()

    scheduler.step()

    avg_train_loss = train_loss / len(train_loader)
    avg_train_dice = train_dice / len(train_loader)

    model.eval()
    val_loss = val_dice = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, masks)

            val_loss += loss.item()
            val_dice += strict_dice_coef(masks, logits).item()

    avg_val_loss = val_loss / len(val_loader)
    avg_val_dice = val_dice / len(val_loader)

    print(f"Train Loss: {avg_train_loss:.4f} | Train Dice: {avg_train_dice:.4f} || Val Loss: {avg_val_loss:.4f} | Val Dice: {avg_val_dice:.4f}")

    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        best_epoch = epoch + 1
        best_model_weights = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), "best_resnet_hmunet.pth")

print("\n--- Training Complete. Evaluating on Unseen Test Set ---")
print(f" Loading Model from Epoch: {best_epoch} ")

model.load_state_dict(best_model_weights)
model.eval()
test_loss = test_dice = 0

with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = criterion(logits, masks)

        test_loss += loss.item()
        test_dice += strict_dice_coef(masks, logits).item()

avg_test_loss = test_loss / len(test_loader)
avg_test_dice = test_dice / len(test_loader)

print(f" Final Test Loss: {avg_test_loss:.4f} | Final Test Dice: {avg_test_dice:.4f}")

Epoch 1/100: 100%|██████████| 130/130 [00:19<00:00,  6.53it/s]


Train Loss: 0.6899 | Train Dice: 0.4070 || Val Loss: 0.5810 | Val Dice: 0.5903


Epoch 2/100: 100%|██████████| 130/130 [00:20<00:00,  6.50it/s]


Train Loss: 0.5766 | Train Dice: 0.5804 || Val Loss: 0.5545 | Val Dice: 0.5645


Epoch 3/100: 100%|██████████| 130/130 [00:20<00:00,  6.44it/s]


...

Train Loss: 0.0954 | Train Dice: 0.9036 || Val Loss: 0.1211 | Val Dice: 0.8809


Epoch 99/100: 100%|██████████| 130/130 [00:19<00:00,  6.67it/s]


Train Loss: 0.0955 | Train Dice: 0.9030 || Val Loss: 0.1251 | Val Dice: 0.8774


Epoch 100/100: 100%|██████████| 130/130 [00:19<00:00,  6.66it/s]


Train Loss: 0.0924 | Train Dice: 0.9067 || Val Loss: 0.1239 | Val Dice: 0.8789

--- Training Complete. Evaluating on Unseen Test Set ---
 Loading Model from Epoch: 90 
 Final Test Loss: 0.1747 | Final Test Dice: 0.8282
